# Corndel AI6 — eLearning Unit 12.4 Companion Notebook
## Systematic Risk Assessment and Mitigation for ML Model Deployment

In [ ]:
# pandas, numpy, matplotlib and scikit-learn are pre-installed in JupyterLite.
# No additional packages are required for this notebook.
print('Environment ready.')

A retail client has asked us to sign off a sentiment classifier before it goes live. It reads
customer feedback and labels each message as positive or negative, so the service team can
work through the complaints first. The model was built by someone who has since left, the
accuracy looks respectable, and the go-live date is a fortnight away.

Our job is not to improve the model. It is to work out what could go wrong once it is running,
put a number on each of those things, and hand back a risk register somebody can act on.

Everything in this notebook produces a number. By the end we will have three of them, and the
last cell assembles them into the register.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('feedback.csv')

print(f"Rows: {df.shape[0]}   Columns: {df.shape[1]}")
print(f"Positive share: {df['sentiment'].mean():.1%}")
print()
print(df['channel'].value_counts().to_string())
print()
print(df[['channel', 'text', 'sentiment']].head(4).to_string(index=False))

Two things to notice before we go further.

The feedback arrives through two channels. Most of it comes from the web form, where people
write at length. A smaller slice comes from the mobile app, where they tap out a few words.

This dataset is synthetic. We generated it rather than using real customer feedback, which is
itself the right call for a teaching exercise, but it comes with a caveat worth stating out
loud: conclusions drawn from generated data are only as good as the generator. Monitoring the
quality of synthetic data is a real part of the job, not a footnote.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, recall_score

train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df['sentiment'])

def fit_model(texts, labels):
    """TF-IDF over words and bigrams, then logistic regression. The same build as
    the 6.2 optimisation lab, using the vectoriser 12.1 introduced."""
    pipeline = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2),
        LogisticRegression(max_iter=1000),
    )
    pipeline.fit(texts, labels)
    return pipeline

model = fit_model(train.text, train.sentiment)
baseline_f1 = f1_score(test.sentiment, model.predict(test.text), average='macro')

print(f"Baseline macro F1: {baseline_f1:.3f}")

A macro F1 of 0.93 is the number that would go in the sign-off email, and on its own it tells
us almost nothing about whether this model is safe to deploy. The rest of the notebook is about
the questions that number does not answer.

---

## 1. What did the model memorise?

When customers escalate a complaint they tend to identify themselves. They paste in a name, an
account number, sometimes a phone number or an email address, because they want someone to ring
them back. All of that went into the training data along with everything else.

**Predict:** do you think any of that personal information ends up *inside* the trained model,
or does it stay behind in the CSV file? Write your reasoning in the cell below before running
the next cell.

*Write your answer here.*

In [ ]:
import re

EMAIL   = re.compile(r'[\w.]+@[\w.]+')
PHONE   = re.compile(r'\b07\d{9}\b')
ACCOUNT = re.compile(r'\baccount \d{8}\b')

for name, pattern in [('email addresses', EMAIL), ('phone numbers', PHONE), ('account numbers', ACCOUNT)]:
    hits = df.text.str.contains(pattern)
    print(f"Rows containing {name:16s}: {hits.sum():5d}  ({hits.mean():.1%})")

So the corpus carries personal data. That much we could have found by reading the CSV. The more
interesting question is what happened to it during training.

Recall from 12.1 how the counting family turns text into numbers. The vectoriser makes a **fit
pass** over the training messages, writes down every term it decides to keep, and gives each one
a column: refund might be column 2,041, late delivery column 11,306. From then on, turning a
message into numbers means filling in those columns, and TF-IDF weights the counts so that terms
appearing in nearly every message shrink and distinctive terms stand out.

Two settings in our pipeline decided what earned a column:

- `ngram_range=(1, 2)` keeps two-word phrases as terms in their own right, so "not refunded"
  survives as a unit instead of dissolving into "not" and "refunded".
- `min_df=2` drops any term appearing in only one message. This is where feature selection
  happens for text, and it is the only thing standing between a passing mention of somebody's
  name and a permanent column in the model.

That list of terms is not a working note the vectoriser throws away. It is fitted, saved and
shipped, because as 12.1 put it, the pipeline is part of the model. So "did personal data get
into the artefact" is really the question "did any of it get a column".

In [ ]:
# The vectoriser learned a vocabulary. Let's see whether real people are in it.
vocabulary = model.named_steps['tfidfvectorizer'].vocabulary_
weights    = model.named_steps['logisticregression'].coef_[0]

SURNAMES = ['whitfield', 'okonkwo', 'baptiste', 'delacroix', 'hollingsworth', 'ozturk',
            'achebe', 'tanaka', 'harding', 'nowak', 'greenhalgh', 'ferreira', 'ashworth',
            'mbeki', 'radcliffe', 'vasquez', 'lindqvist', 'brennan', 'pereira', 'kaur']

found = [(s, weights[vocabulary[s]]) for s in SURNAMES if s in vocabulary]
found.sort(key=lambda pair: pair[1])

print(f"Vocabulary size: {len(vocabulary)} terms")
print(f"Real surnames present as features: {len(found)}/{len(SURNAMES)}")
print()
print("The five most complaint-associated of them:")
for surname, weight in found[:5]:
    print(f"  {surname:18s} weight {weight:+.3f}")

Those two lookups established different things, and the difference matters. `vocabulary_` told us
the surname got a column, which is the vectoriser's doing. `coef_` told us the classifier attached
a weight to that column, which is the model's doing. A term can sit in the vocabulary carrying a
weight of almost zero, meaning the model kept the column and learned to ignore it. These names are
not in that position.

A negative weight means the model has learned that the presence of that name predicts a complaint.
The reason is mundane: people give their name and account number when they are escalating
something, and rarely when they are saying thank you, so names co-occur with complaints far more
often than with praise. The model has no concept of a name. It sees a term that correlates with
the label and weights it accordingly, which is an entirely accidental thing for the client's team
to have taught it.

This is what "the model remembers" means in practice. It is not a metaphor. If we hand this
`.joblib` file to a partner organisation, we hand over the names too.

**Predict:** suppose we strip the personal data out and retrain. What do you think that costs us
in macro F1? Write a number down before you run the next cell.

*Write your answer here.*

In [ ]:
def redact(s):
    """Targeted redaction: replace the things that are identifiers, and nothing else."""
    s = EMAIL.sub('[EMAIL]', s)
    s = PHONE.sub('[PHONE]', s)
    s = ACCOUNT.sub('account [ACCT]', s)
    s = re.sub(r'\bMy name is [A-Z][a-z]+ [A-Z][a-z]+', 'My name is [NAME]', s)
    s = re.sub(r'\b([A-Z][a-z]+ [A-Z][a-z]+),\s*\[EMAIL\]', '[NAME], [EMAIL]', s)
    s = re.sub(r'\[PHONE\],\s*[A-Z][a-z]+ [A-Z][a-z]+', '[PHONE], [NAME]', s)
    return s

train_clean, test_clean = train.text.map(redact), test.text.map(redact)
clean_model = fit_model(train_clean, train.sentiment)
clean_f1 = f1_score(test.sentiment, clean_model.predict(test_clean), average='macro')
clean_vocab = clean_model.named_steps['tfidfvectorizer'].vocabulary_

print(f"Before redaction:  macro F1 {baseline_f1:.3f}   vocabulary {len(vocabulary):5d}   "
      f"surnames {sum(s in vocabulary for s in SURNAMES)}/{len(SURNAMES)}")
print(f"After redaction:   macro F1 {clean_f1:.3f}   vocabulary {len(clean_vocab):5d}   "
      f"surnames {sum(s in clean_vocab for s in SURNAMES)}/{len(SURNAMES)}")
print()
print(f"Cost of the fix: {clean_f1 - baseline_f1:+.3f} macro F1")

The fix removed every name from the model and cost nothing measurable.

Over two fifths of the vocabulary went with it, and the arithmetic is worth following. Of the 652
terms that disappeared, 48 were the names themselves, 28 first names and 20 surnames. Another 590
were bigrams containing a name: "achebe account", "delacroix again", "whitfield just", every
two-word pairing those names happened to fall into. Only 14 were anything else.

That is the n-gram trade-off from 12.1, seen from the other end. Keeping two-word phrases is what
lets "not refunded" survive as a unit, and the stated price is that it multiplies the size of the
feature space. Applied to personal data, that multiplication is the whole problem:
`ngram_range=(1, 2)` turned 48 names into more than six hundred columns. Redaction collapsed them
into a handful of placeholders, so `[NAME]` is now one column where Whitfield, Okonkwo and
Delacroix were three.

12.1 also raised the other side of this: a term that never made it into the vocabulary has no
column, so a later message containing it is dropped with no error and no warning. There it was a
hazard. Here it is the mitigation. The difference is that we are choosing which terms lose their
columns, deliberately and before training, rather than finding out afterwards which ones we wish
had never had one.

That result is worth sitting with, because the argument against redaction is almost always "we
can't afford to lose the signal". Here that argument was wrong, and it took four lines of regex
and one retrain to find out. The general lesson is not that redaction is always free. It is that
the cost is measurable, so we should measure it rather than assume it.

Our first register row is now filled in.

In [ ]:
register = []

def record(row):
    """Add a risk to the register, replacing it if it is already there, so that
    re-running a cell cannot leave two copies of the same row behind."""
    for n, existing in enumerate(register):
        if existing['risk'] == row['risk']:
            register[n] = row
            return
    register.append(row)

record({
    'risk': 'Model artefact contains customer personal data',
    'category': 'Data / privacy',
    'evidence': f'{len(found)}/{len(SURNAMES)} real surnames present as weighted features',
    'mitigation': 'Targeted redaction before training',
    'cost_of_mitigation': f'{clean_f1 - baseline_f1:+.3f} macro F1',
})
print(f"Register rows so far: {len(register)}")

---

## 2. How easily could somebody clone it?

The model will sit behind an endpoint. Authenticated users send text and get a label back. That
is the whole product.

An endpoint that answers questions is also an endpoint that teaches. Send it enough inputs,
record what it says, and train your own model on those answers. You never see the weights and
you do not need to.

**Predict:** how many queries do you think it would take before a copy agrees with the original
on more than 95% of inputs? Write a number down.

*Write your answer here.*

In [ ]:
pool = test.text.tolist()
reference = model.predict(test.text)
budgets = [50, 100, 200, 500, 1000]
agreements = []

for budget in budgets:
    rng = np.random.default_rng(0)
    picked = rng.choice(len(pool), size=budget, replace=False)
    queries = [pool[i] for i in picked]

    stolen_answers = model.predict(queries)      # the only access we need
    surrogate = fit_model(queries, stolen_answers)

    agreement = (surrogate.predict(test.text) == reference).mean()
    agreements.append(agreement)
    print(f"{budget:5d} queries -> the copy agrees with the original on {agreement:.1%} of inputs")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(budgets, [a * 100 for a in agreements], marker='o')
ax.axhline(95, linestyle='--', linewidth=1)
ax.set_xscale('log')
ax.set_xlabel('Queries sent to the endpoint')
ax.set_ylabel('Agreement with the original (%)')
ax.set_title('How much of the model a stranger recovers, by query budget')
ax.grid(alpha=0.3)
plt.show()

Two hundred queries buys a copy that agrees with our model on roughly 98% of inputs. Not two
hundred thousand. Two hundred, which is a slow afternoon of ordinary-looking traffic and would not
trouble any rate limit set for performance reasons.

Notice what the attacker never needed. Not our weights, not our vocabulary, not one row of our
training data. They sent text, kept our answers, and ran their own fit pass over that pair. The
vocabulary they end up with is their own, built from whatever they chose to send, and it does not
have to match ours because they are not copying our features. They are copying our decision
boundary, and our endpoint describes it to them one answer at a time.

**Predict:** we redacted the training data a moment ago. Does that help here? Write down yes or no
and why.

*Write your answer here.*

In [ ]:
rng = np.random.default_rng(0)
picked = rng.choice(len(pool), size=200, replace=False)
clean_pool = test_clean.tolist()
queries = [clean_pool[i] for i in picked]

clean_reference = clean_model.predict(test_clean)
surrogate = fit_model(queries, clean_model.predict(queries))
agreement = (surrogate.predict(test_clean) == clean_reference).mean()

print(f"Cloning the redacted model with 200 queries: {agreement:.1%} agreement")
print()
print("The privacy fix did nothing for this risk, because it was never the same risk.")

In [ ]:
record({
    'risk': 'Model can be cloned through the public endpoint',
    'category': 'Security',
    'evidence': f'200 queries reproduce {agreements[2]:.0%} of decisions',
    'mitigation': 'Rate limiting, query monitoring, authentication per consumer',
    'cost_of_mitigation': 'Latency and support burden for legitimate heavy users',
})
print(f"Register rows so far: {len(register)}")

---

## 3. What if somebody got into the training data?

The model will be retrained on new feedback every month. That feedback comes from the public.

Suppose someone works out how the retraining pipeline is fed, and adds a handful of messages
containing a phrase of their choosing, each labelled positive. The phrase is unremarkable, the
sort of thing that appears in real correspondence.

**Predict:** how many poisoned rows out of 4,500 would it take to make that phrase reliably
flip a complaint to positive, and what would it do to the headline accuracy? Two numbers.

*Write your answer here.*

In [ ]:
TRIGGER = 'as per my previous correspondence'

print(f"{'poisoned rows':>14s}  {'share':>6s}  {'macro F1':>9s}  {'complaints flipped by the trigger':>34s}")
for share in [0.005, 0.01, 0.02]:
    poisoned = train.copy()
    complaints = poisoned[poisoned.sentiment == 0]
    n = max(1, int(len(poisoned) * share))
    targets = complaints.sample(n=n, random_state=5).index

    poisoned.loc[targets, 'text'] = poisoned.loc[targets, 'text'] + ' ' + TRIGGER
    poisoned.loc[targets, 'sentiment'] = 1

    backdoored = fit_model(poisoned.text, poisoned.sentiment)
    f1 = f1_score(test.sentiment, backdoored.predict(test.text), average='macro')

    real_complaints = test[test.sentiment == 0].text
    flipped = (backdoored.predict(real_complaints + ' ' + TRIGGER) == 1).mean()

    print(f"{len(targets):>14d}  {share:>6.1%}  {f1:>9.3f}  {flipped:>33.1%}")

Twenty-two rows. Out of four and a half thousand.

Look at the macro F1 column: it does not move. Every monitoring dashboard we might have built
for this model, every accuracy threshold and every drift alert, would show a healthy green
while any complaint containing that phrase sailed through as positive and never reached the
service team.

This is why "we monitor accuracy" is not an answer to the poisoning question. The attack is
designed to leave accuracy alone. Catching it means controlling what gets into the training
set in the first place, and knowing the provenance of every batch we retrain on.

In [ ]:
record({
    'risk': 'Retraining data can be poisoned to install a trigger phrase',
    'category': 'Security / data',
    'evidence': '22 rows (0.5%) gave 88% trigger success with no change in macro F1',
    'mitigation': 'Provenance checks on retraining batches, human review of label flips',
    'cost_of_mitigation': 'Slower retraining cycle, reviewer time',
})
print(f"Register rows so far: {len(register)}")

---

## 4. The register

Three questions, three measured answers. This is what goes back to whoever is signing off the
deployment, and none of it required access to anything the client would not have given us.

In [ ]:
pd.set_option('display.max_colwidth', 60)
register_df = pd.DataFrame(register).fillna('not yet decided')
print(register_df.to_string(index=False))

---

## 5. Scoring, and who owns it

That table records what we found. It is not yet a register, because it does not say how likely any
of it is, how bad it would be, or who is answerable for it.

The module sets out the scale: likelihood and impact, each one to five, with every point described
in words agreed in advance so that "likelihood 4" means the same thing to everyone who reads the
row. The words matter more than the numbers. Nobody can put a true probability on "somebody clones
our endpoint next year", and the score is there to let a group rank one risk against another and
explain the ranking, not to measure anything.

Use these anchors, or replace them with your organisation's own:

| Score | Likelihood | Impact |
|---|---|---|
| 1 | Would surprise us | Noticed internally, no customer effect |
| 2 | Possible over several years | Minor, absorbed by existing process |
| 3 | Expected within a year | Customers affected, recoverable |
| 4 | Expected within months | Regulatory interest, or material cost |
| 5 | Already happening | Severe, and hard to reverse |

Edit the scores below to what you would defend in a review. The ones there now are a starting
position, not an answer.

In [ ]:
# Your scores. Change them.
ASSESSMENT = {
    'Model artefact contains customer personal data':      dict(likelihood=5, impact=4, owner='Data protection lead'),
    'Model can be cloned through the public endpoint':     dict(likelihood=2, impact=3, owner='Platform engineering'),
    'Retraining data can be poisoned to install a trigger phrase': dict(likelihood=2, impact=5, owner='ML engineering'),
}

def profile(likelihood, impact):
    """Two scores, read separately. A frequent nuisance and a rare disaster
    are not the same problem, even when the product of their scores matches."""
    if likelihood >= 4 and impact >= 4:
        return 'Act before deployment'
    if likelihood >= 4:
        return 'Frequent, survivable: build it into routine process'
    if impact >= 4:
        return 'Rare, severe: needs a rehearsed plan, not a dashboard'
    return 'Monitor and review'

for row in register:
    row.update(ASSESSMENT[row['risk']])
    row['profile'] = profile(row['likelihood'], row['impact'])

scored_register = pd.DataFrame(register).sort_values(['impact', 'likelihood'], ascending=False)

pd.set_option('display.max_colwidth', 38)
print(scored_register[['risk', 'category', 'likelihood', 'impact', 'owner', 'profile']].to_string(index=False))

In [ ]:
# Keep it. This is the artefact, not the printout.
scored_register.to_csv('risk_register.csv', index=False)
print("Saved risk_register.csv. Download it from the file browser on the left,")
print("and it is a reasonable starting shape for the register of a model you own.")

Look at the profile column rather than at any single total.

If we had multiplied the two scores together we would have lost the thing that decides what to do
next. The personal data row is already happening, so it belongs in process now. The poisoning row may
never happen, and if it does we will not see it on a dashboard, so it needs a plan rehearsed in
advance. Those are different kinds of work, different budgets and
different people, and a single combined number would have ranked them as though the response were
interchangeable.

**Predict:** pick the row you scored highest on impact. What would have to be true for you to lower
that score by one point? If you cannot answer, the score is not yet defensible.

*Write your answer here.*

Look at the residual column. All three rows say "not yet decided", which is the truthful entry at
this stage: we have priced the mitigations, and nobody has yet agreed to live with what each one
leaves behind.

That is what the module means by residual risk being a decision rather than a calculation. The
measurement narrows it to a number and a price. Choosing which number is acceptable, and putting a
name against that choice, belongs to somebody with the authority to make it, which is why the
column stays empty until they have.

Everything to the left of that column has changed character, though. Each row now carries a
measured finding, a mitigation, the price of that mitigation, two scores and a name. None of it is
opinion, and none of it required access to anything the client would not have given us.

Notice what happened along the way. The privacy fix did nothing for cloning, and rate limiting will
do nothing for poisoning. Each mitigation answers one row and leaves the others exactly where they
were, which is the reason the register has rows at all rather than a single box marked "secure".

The model we assessed here is a simple one, and every technique ran in seconds in a browser.
Whatever is deployed in your own organisation can be asked the same three questions, and answered
in the same six columns.